In [ ]:
import os

os.environ["TORCH_HOME"] = "/aiau010_scratch/azm0269/clover/.cache/torch"
os.environ["HF_HOME"] = "/aiau010_scratch/azm0269/hub"

import gc
# import torch

gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

os.listdir(); os.chdir("/aiau010_scratch/azm0269/clover/")

from clover.utils.utils import notebook_line_magic
notebook_line_magic()

In [2]:
from pathlib import Path

baselines = ["ddpo", "b2diffurl", "emo_v3", "sd15"]  # sd15 = vanilla Stable Diffusion 1.5
checkpoint_path = Path.cwd() / "outputs"


### B2 evaluation prompts by template

Use the held-out evaluation split from `clover.utils.prompts`: template 1 covers object behavior (45 prompts), template 2 object attributes (10), and template 3 positional relationships (10). Pass a list directly to `model.generate(...)`, or select it with `prompts_by_template["object_behaviour"]`.


### Load a baseline for downstream inference

`load_baseline_model(baseline)` restores the base model, checkpoint LoRA weights, and (for EMO v2/v3/v4, including v3_old) the separate FP32 variance head. Defaults to `outputs/<baseline>/seed_123/checkpoint`; pass `run_dir` for another run. Only load trusted project checkpoints: training payloads contain pickled RNG state.

Use `model.generate(...)` for PIL images. EMO uses its training implementation of learned-variance sampling and noise-only classifier-free guidance. `model.pipe` exposes the underlying pipeline for inspection; use `generate` for variance-aware inference. Run the B2 evaluation stages below in the notebook.


In [33]:
from clover.utils.prompts import (
    TEMPLATE_1_EVAL_PROMPTS,
    TEMPLATE_2_EVAL_PROMPTS,
    TEMPLATE_3_EVAL_PROMPTS,
)
from clover.evaluate.inference import LoadedBaseline, load_baseline_model
from clover.evaluate.b2_run_evaluation import run_b2_evaluation

# Held-out B2 evaluation prompts, grouped by template number.
template_1_object_behavior_prompts = list(TEMPLATE_1_EVAL_PROMPTS)
template_2_object_attribute_prompts = list(TEMPLATE_2_EVAL_PROMPTS)
template_3_positional_relationship_prompts = list(TEMPLATE_3_EVAL_PROMPTS)

prompts_by_template = {
    "object_behaviour": template_1_object_behavior_prompts,
    "object_attribute": template_2_object_attribute_prompts,
    "positional_relationship": template_3_positional_relationship_prompts,
}


In [ ]:
b2_run_name = "b2_four_baselines_v2"
run_b2_evaluation(
    baselines=baselines,
    seeds=[123, 124, 126],
    b2_run_name=b2_run_name,
    images_per_prompt=10,
    fraction=0.8,
)


### T2I-CompBench (in addition to B2)

The setup, generation, and metric functions are imported from `clover.evaluate.t21_comp_bench`. Run the import cell first, then the setup and evaluation cells. They do not submit jobs or require a Slurm job ID.

The official validation sets each contain 300 prompts. **BLIP-VQA** scores color, shape, and texture binding; **UniDet** scores spatial relationships. Non-spatial and complex prompts are also available, but their CLIPScore/3-in-1 metrics are not included. [Official source](https://github.com/Karine-Huang/T2I-CompBench).

The legacy metric dependencies use the local uv project in `clover/evaluate/compbench_backend`, separate from the notebook's model-generation environment. First-time runtime setup requires a compatible CUDA 11.7 toolchain for Detectron2; BLIP weights download on first scoring. Only one visible GPU is exposed to each scorer.


In [ ]:
from clover.evaluate.t21_comp_bench import (
    setup_compbench,
    load_compbench_prompts,
    generate_compbench_images,
    run_compbench_metrics,
    collect_compbench_metrics,
)


### One-time benchmark setup

The imported `setup_compbench()` function downloads the pinned official source and prompts. Use `setup_compbench(install_runtime=True)` once to install metric dependencies and the UniDet weights from this notebook. Skip that installation call if the metric runtime is already prepared.


In [ ]:
# Fetch/verify source and prompts. This step uses no GPU.
compbench_source = setup_compbench()

# Run once to install the metric runtime and UniDet weights:
# setup_compbench(install_runtime=True)

compbench_prompts = load_compbench_prompts()
compbench_color_prompts = compbench_prompts["color"]
compbench_shape_prompts = compbench_prompts["shape"]
compbench_texture_prompts = compbench_prompts["texture"]
compbench_spatial_prompts = compbench_prompts["spatial"]
{category: len(prompts) for category, prompts in compbench_prompts.items()}


### Generate from a loaded checkpoint

Use the `model` loaded with `load_baseline_model(...)` above. Set a new run name for each evaluation. The default below scores all four supported categories with ten images per prompt (12,000 images). For a quick check, pass `prompts_by_category={"color": compbench_color_prompts[:2]}` and `images_per_prompt=1`.


In [ ]:
from IPython.utils.capture import capture_output

with capture_output():
    # Load a model for the separate CompBench stage after B2 releases its models.
    model = load_baseline_model("emo_v3", output_root=checkpoint_path)
    
    manifest_path = generate_compbench_images(
        model,
        run_name="compbench_v1",
        images_per_prompt=10,
        base_seed=123,
        inference_kwargs={"num_inference_steps": 50, "guidance_scale": 7.5},
    )


### Score the generated images

Offload the checkpoint to CPU to leave GPU memory for the metric models. BLIP-VQA and UniDet run sequentially; the notebook waits for them and then displays category summaries. Images and seed metadata are under `clover/data/<baseline>/t2i_compbench/<run_name>/`; structured scores are under `outputs/<baseline>/evals/t2i_compbench/<run_name>/metrics.json`.


In [ ]:
import torch

model.pipe.to("cpu")
gc.collect()
torch.cuda.empty_cache()

results = run_compbench_metrics(manifest_path)
results["summary"]

# Before generating again with this model:
# model.pipe.to(model.device)
# if model.variance_module is not None:
#     model.variance_module._variance_head(model.pipe).float()


In [ ]:
# Reload/aggregate existing scores without running the metric models:
# results = collect_compbench_metrics(manifest_path)
# results["summary"]
